# Part 1: 데이터 & 간단 평가셋 만들기

1. **데이터 소개**: 'IBK 증권보고서_삼성전자_SK하이닉스' PDF 구조
2. **질문 리스트 설계**: 핵심 요약/수치 질의/표 기반 질의 등 카테고리

## 1-1. 데이터 소개: 'IBK 증권보고서_삼성전자_SK하이닉스' PDF 구조

이 PDF는 IBK투자증권이 작성한 삼성전자·SK하이닉스 리포트입니다. RAG 실험에 적합한 이유:
- **목차**: 삼성전자/SK하이닉스 실적, Check Point, 투자의견 등 구조화된 구성
- **표**: 분기별 실적 추이, 사업부별 매출/영업이익 등 표가 많음 (행·열 구조 중요)
- **주석·각주**: 단위(십억원, 조원), 출처 등
- **페이지 구성**: 26페이지

In [27]:
# PDF 기본 정보 확인
import fitz
pdf_path = "IBK_증권보고서_삼성전자_SK하이닉스.pdf"

doc = fitz.open(pdf_path)
print(f"페이지 수: {len(doc)}")

page6 = doc[6]
text = page6.get_text()[:1000]
print(text)

doc.close()

페이지 수: 26
 
김운호  6915-5656  
 
 
IBKS RESEARCH │7 
 2025년 4분기 매출액은 93.8조원 
삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. 
MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 가격 상승
으로 32.9% 증가, VD/가전은 연말 성수기 효과로 6.1% 증가, MX 사업부는 물량은 
소폭 감소, ASP 하락 영향이 커서 14.1% 감소, 디스플레이는 중소형, 대형 모두 물량
이 개선된 영향으로 17.4% 증가했다. 사업부별로는  
1) DS 사업부 매출액은 2025년 3분기 대비 32.9% 증가한 44조원이다. 메모리는 
25년 3분기 대비 38.9% 증가했다. 2025년 4분기 DRAM B/G(Bit Growth)는 
+2.0%, ASP는 +39.5%로 추정한다. NAND B/G는 -10%, ASP는 +25.0%로 
추정한다. DRAM ASP 상승은 Conventional 제품 가격 상승과 제품믹스 
개선에 따른 영향이다. NAND는 eSSD 수요 확대로 가격이 큰 폭으로 
상승하였다. 비메모리는 7.9% 증가했다. 파운드리 물량 증가 영향이다.  
2) 디스플레이 사업부 매출액은 2025년 3분기 대비 17.4% 증가한 9.5조원이다. 
해외 물량 성수기 영향과 국내 고객 물량의 조기 생산 영향, IT/Wearable 물량 
증가로 3분기 대비 증가하였다. 대형 물량도 3분기 대비 증가했다. 
3) MX/네트워크 
사업부 
매출액은 
2025년 
3분기 
대비 
15.6% 
감소한 
29.3조원이다. MX는 15.6% 감소하였다. 물량이 감소하고, ASP도 하락한 
영향이다. 네트워크 사업부 매출액은 전분기 대비 70% 증가했다.  
4) VD/가전 사업부 매출액은 2025년 3분기 대비 6.1% 증가한 14.8조원이다.  
VD 매출액은 25년 3분기 대비 20.5% 증가했고, 가전은 2025년 3분기 대비 
9.8% 감소했다. 

## 1-2. 기본 질문 및 정답 리스트 설계

RAG 평가를 위한 질문은 다음 유형으로 설계할 수 있습니다:
- **핵심 요약**: "삼성전자 2025년 4분기 실적 요약은?"
- **수치 질의**: "매출액", "영업이익", "목표주가" 등 구체적 수치
- **표 기반 질의**: 표에서 특정 행·열 교차점 값을 묻는 질문
- **리스크 질의**: 투자 리스크, 전망 등

본 실험에서는 **재무(수치)**, **투자의견**, **비교** 카테고리의 12개 질문을 사용합니다.

평가 항목:
- **정확도**: expected_answer가 LLM 답변에 포함되는지
- **근거성**: 인용된(retrieved) 문서에 expected_answer 또는 관련 수치가 포함되는지 (Yes/No)
- **속도**: 응답 시간(초)

In [28]:
# 기존 12개 질문의 카테고리 분포
from collections import Counter

test_questions = [
    {"question": "삼성전자 2025년 4분기 매출액은?", "keywords": ["삼성전자", "2025", "4분기", "매출액"], "expected_answer": "93.8조원", "category": "재무"},
    {"question": "삼성전자 2025년 4분기 영업이익은?", "keywords": ["삼성전자", "2025", "4분기", "영업이익"], "expected_answer": "20.1조원", "category": "재무"},
    {"question": "삼성전자 DS 부문 2025년 4분기 영업이익은?", "keywords": ["DS", "영업이익", "2025"], "expected_answer": "16.4조원", "category": "재무"},
    {"question": "삼성전자 메모리 사업부 2025년 4분기 영업이익은?", "keywords": ["메모리", "영업이익", "2025"], "expected_answer": "17.2조원", "category": "재무"},
    {"question": "SK하이닉스 2025년 4분기 영업이익은?", "keywords": ["SK하이닉스", "영업이익", "2025"], "expected_answer": "19.17조원", "category": "재무"},
    {"question": "2026년 1분기 삼성전자 영업이익은?", "keywords": ["2026", "삼성전자", "영업이익"], "expected_answer": "35.3조원", "category": "재무"},
    {"question": "2026년 1분기 SK하이닉스 영업이익은?", "keywords": ["2026", "SK하이닉스", "영업이익"], "expected_answer": "31조원", "category": "재무"},
    {"question": "삼성전자 목표주가는?", "keywords": ["삼성전자", "목표주가"], "expected_answer": "240,000원", "category": "투자의견"},
    {"question": "SK하이닉스 목표주가는?", "keywords": ["SK하이닉스", "목표주가"], "expected_answer": "1,100,000원", "category": "투자의견"},
    {"question": "25년 4분기 영업이익은 삼성전자와 SK하이닉스 중 누가 더 큰가?", "keywords": ["25년", "4분기", "영업이익"], "expected_answer": "삼성전자", "category": "비교"},
    {"question": "2026년 1분기 삼성전자 메모리 부문 영업이익은?", "keywords": ["2026", "메모리", "영업이익"], "expected_answer": "33.1조원", "category": "재무"},
    {"question": "2025년 4분기 DRAM ASP 상승률은 삼성과 SK 중 어느 쪽이 더 높은가?", "keywords": ["DRAM", "ASP", "삼성"], "expected_answer": "삼성전자", "category": "비교"},
]

cats = Counter(t["category"] for t in test_questions)
for c, n in cats.items():
    print(f"{c}: {n}개")

print(f"테스트 질문 수: {len(test_questions)}")

재무: 8개
투자의견: 2개
비교: 2개
테스트 질문 수: 12


# Part 2: PDF 추출 방식 비교 + Base RAG

1. **이론**: 왜 "추출"이 RAG 성능을 좌우하는가
2. **PDF 추출 파이프라인 3종**: PyMuPDFLoader, PyMuPDF4LLMLoader, VLM
3. **Base RAG 세팅**: chunk/embedding/LLM 고정
4. **FAISS 기반 답변 비교**: 3종 추출 결과물 확인

## 2-1. 이론: 왜 "추출"이 RAG 성능을 좌우하는가

PDF는 원래 시각적 레이아웃을 위한 포맷입니다. 텍스트만 뽑으면 문제가 생깁니다.

- **레이아웃**: 2단 구성, 페이지 나눔 등으로 문장 순서가 뒤바뀔 수 있음
- **표**: 셀 경계가 무시되면 "25,126 | 27,935"가 "25 126 27 935"처럼 깨짐
- **헤더/푸터**: 매 페이지 반복되는 헤더가 노이즈가 됨
- **줄바꿈/하이픈**: "매출액\n93.8조"가 한 단위로 인식되지 않을 수 있음

따라서 추출 방식을 바꾸면 검색 품질과 최종 답변 정확도가 달라집니다.

In [29]:
import os  
import io
import time  
import pickle  
from typing import Optional, List 

import torch
from PIL import Image
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Qwen2_5_VLProcessor,
    Qwen2_5_VLForConditionalGeneration,
    pipeline,
 )

from langchain_community.document_loaders import PyMuPDFLoader  # PDF 텍스트 로더
from langchain_pymupdf4llm import PyMuPDF4LLMLoader  # PDF를 LLM  로드
from langchain_core.documents import Document  # 문서 객체
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 문서 청크 분할기
from langchain_community.vectorstores import FAISS  # 벡터 저장소
from langchain_community.embeddings import HuggingFaceEmbeddings  
from dotenv import load_dotenv  
load_dotenv()  

True

## 2-2. 추출 파이프라인 3종 구현

In [30]:
# (1) PyMuPDFLoader — 텍스트 중심 추출
loader_pymupdf = PyMuPDFLoader(pdf_path)
docs_pymupdf = loader_pymupdf.load()
print(f"PyMuPDFLoader: {len(docs_pymupdf)}개 페이지")

PyMuPDFLoader: 26개 페이지


In [32]:
# (2) PyMuPDF4LLMLoader — 레이아웃·LLM 친화 Markdown
loader_4llm = PyMuPDF4LLMLoader(pdf_path)
docs_4llm = loader_4llm.load()
print(f"PyMuPDF4LLMLoader: {len(docs_4llm)}개 페이지")

Performing OCR on page.number=0[1]...
PyMuPDF4LLMLoader: 26개 페이지


In [33]:
# (3) VLM 기반 추출 — 이미지 → Qwen2.5-VL(Hugging Face 로컬 로딩) → Markdown
import fitz  # PDF를 열고 페이지를 이미지로 변환하는 라이브러리

# PDF의 각 페이지를 이미지(PNG 바이트)로 변환하는 함수
def pdf_to_images(pdf_path: str, max_pages: Optional[int] = None, start_page: int = 0) -> List[tuple]:
    doc = fitz.open(pdf_path)  
    images = [] 
    end = min(start_page + (max_pages or len(doc)), len(doc))  
    for i in range(start_page, end):  
        page = doc[i]  
        pix = page.get_pixmap(dpi=200)  
        images.append((i + 1, pix.tobytes("png")))  
    doc.close()  
    return images  

# 페이지 이미지를 VLM에 보내 텍스트/표/차트 등을 Markdown으로 추출하는 함수
def vlm_extract_page(image_bytes: bytes, processor, model) -> str:
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    prompt = (
        "You are a document OCR assistant for educational purposes. "
        "Please extract all visible text, tables, charts, and graphs from this image "
        "and convert them into well-structured Markdown format. "
        "Use | delimiters for Markdown tables. Accurately include all numbers and units. "
        "Output in the same language as the document."
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text], images=[image], return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=1024)

    generated_ids_trimmed = generated_ids[:, inputs.input_ids.shape[1]:]
    response = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    return response

def load_pdf_with_vlm(pdf_path: str, processor, model, max_pages: Optional[int] = None, start_page: int = 0):
    images = pdf_to_images(pdf_path, max_pages=max_pages, start_page=start_page)
    docs = []
    for page_num, img_bytes in images:
        text = vlm_extract_page(img_bytes, processor, model)
        docs.append(Document(page_content=text, metadata={"source": pdf_path, "page": page_num}))
    return docs

In [34]:
import os
import time

from transformers import Qwen2_5_VLProcessor, Qwen2_5_VLForConditionalGeneration

HF_TOKEN = os.getenv("HF_TOKEN")
QWEN_VL_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"

# Hugging Face Xet 경로가 느릴 때 다운로드가 멈춘 것처럼 보일 수 있어 비활성화
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

# 토큰은 공개 모델이면 없어도 동작함 (있으면 전달)
hf_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}

# VLM 로드 (처음 1회만 오래 걸릴 수 있음)
vlm_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
load_start = time.time()
print("[1/2] loading processor...")
vlm_processor = Qwen2_5_VLProcessor.from_pretrained(
    QWEN_VL_MODEL,
    trust_remote_code=True,
    **hf_kwargs,
)
print(f"[1/2] processor loaded in {time.time() - load_start:.2f}s")

print("[2/2] loading model...")
model_start = time.time()
vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_VL_MODEL,
    torch_dtype=vlm_dtype,
    device_map="auto",
    trust_remote_code=True,
    **hf_kwargs,
)
print(f"[2/2] model loaded in {time.time() - model_start:.2f}s")
print(f"VLM model ready: {QWEN_VL_MODEL}")
print(f"HF token provided: {'yes' if HF_TOKEN else 'no'}")

[1/2] loading processor...
[1/2] processor loaded in 4.78s
[2/2] loading model...


Loading weights: 100%|██████████| 729/729 [00:03<00:00, 217.92it/s]


[2/2] model loaded in 5.11s
VLM model ready: Qwen/Qwen2.5-VL-7B-Instruct
HF token provided: yes


In [35]:
# VLM 1페이지 스모크 테스트
smoke_start = time.time()
print("[smoke] running 1-page extraction...")
docs_vlm = load_pdf_with_vlm(
    pdf_path=pdf_path,
    processor=vlm_processor,
    model=vlm_model,
    max_pages=1,
    start_page=0,
)

elapsed = time.time() - smoke_start
print(f"VLM: {len(docs_vlm)}개 페이지")
print(f"VLM smoke test 소요 시간: {elapsed:.2f}초")

[smoke] running 1-page extraction...
VLM: 1개 페이지
VLM smoke test 소요 시간: 5.42초


In [36]:
# VLM 전체 페이지 추출 (Qwen2.5-VL, 모델 로드 완료 후 실행)
start_time = time.time()
docs_vlm = load_pdf_with_vlm(
    pdf_path=pdf_path,
    processor=vlm_processor,
    model=vlm_model,
    max_pages=None,
    start_page=0,
)
elapsed = time.time() - start_time
print(f"VLM: {len(docs_vlm)}개 페이지")  # 추출된 페이지 수 출력
print(f"VLM 추출 소요 시간: {elapsed:.2f}초")


VLM: 26개 페이지
VLM 추출 소요 시간: 304.69초


### 추출 결과 비교: 동일 페이지에서 3종 로더의 차이

표가 포함된 페이지(9페이지)를 기준으로, 각 로더가 어떻게 다르게 추출하는지 비교합니다.
- **PyMuPDFLoader**: 텍스트만 추출 → 표 구조 손실 가능
- **PyMuPDF4LLMLoader**: Markdown 형태로 표 구조 보존 시도
- **VLM (Qwen2.5-VL)**: 이미지 인식 → 표/차트까지 Markdown으로 변환


In [37]:
# 9페이지(0-based index 8) 비교 — 표가 포함된 페이지
page_idx = 8
char_limit = 800

print("=" * 80)
print("PyMuPDFLoader - 페이지 {} 추출 결과 (앞 {}자)".format(page_idx + 1, char_limit))
print("=" * 80)
print(docs_pymupdf[page_idx].page_content[:char_limit])

print("\n" + "=" * 80)
print("PyMuPDF4LLMLoader - 페이지 {} 추출 결과 (앞 {}자)".format(page_idx + 1, char_limit))
print("=" * 80)
print(docs_4llm[page_idx].page_content[:char_limit])

has_table_md = "|" in docs_4llm[page_idx].page_content
print("\nMarkdown 테이블 구분자 포함: {}".format("예" if has_table_md else "아니오"))

vlm_page_idx = min(page_idx, len(docs_vlm) - 1)
vlm_page_num = docs_vlm[vlm_page_idx].metadata.get("page", vlm_page_idx + 1)
print("\n" + "=" * 80)
print("[VLM] Qwen2.5-VL - 페이지 {} 추출 결과 (앞 {}자)".format(vlm_page_num, char_limit))
print("=" * 80)
print(docs_vlm[vlm_page_idx].page_content[:char_limit])
has_vlm_table = "|" in docs_vlm[vlm_page_idx].page_content
print("\n[VLM] Markdown 테이블 구분자 포함: {}".format("예" if has_vlm_table else "아니오"))


PyMuPDFLoader - 페이지 9 추출 결과 (앞 800자)
김운호  6915-5656  
 
 
IBKS RESEARCH │9 
표 1. 삼성전자 분기별 실적 추이 및 전망 
 
(단위: 십억원) 
2025 
2026 
4분기 증감률 
1Q 
2Q 
3Q 
4Q 
1QE 
2QE 
3QE 
4QE 
QoQ(%) 
YoY(%) 
매출액 
DS 
25,126 
27,935 
33,098 
43,983 
56,633 
67,768 
73,008 
74,604 
32.9 
46 
 
Display 
5,931 
6,419 
8,119 
9,528 
6,230 
6,509 
7,991 
7,611 
17.4 
17.5 
 
MX/네트워크 
37,058 
29,208 
34,147 
29,329 
30,457 
25,443 
28,674 
25,346 
-14.1 
13.8 
 
VD/가전 
14,509 
13,882 
13,952 
14,804 
13,150 
13,613 
13,279 
13,504 
6.1 
2.8 
 
HAR 
3,409 
3,818 
4,009 
4,610 
4,841 
5,083 
5,337 
5,604 
15 
17.7 
 
합계 
79,141 
74,566 
86,062 
93,865 
103,089 
110,358 
120,392 
118,931.00 
9.1 
23.9 
영업이익 
DS 
1,128 
405 
6,988 
16,383 
32,154 
42,466 
47,710 
48,694 
134.4 
462.7 
 
Display 
525 
506 
1,207 
2,012 
514 
612 
1,253 
1,155 
66.7 
123.2 


PyMuPDF4LLMLoader - 페이지 9 추출 결과 (앞 800자)
김운호  6915-5656 

## 표 1. 삼성전자 분기별 실적 추이 및 전망 

|(단위: 십억원)|(단위: 십억원)|2025|2026|4분기 증감률|
|---|---|---|---|---|
|||1Q<br>2Q

In [38]:
# 3종 추출본에 동일 TextSplitter 적용
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=180,
    separators=["\n\n", "\n", ". ", ",", " "],
    length_function=len,
)

splits_pymupdf = text_splitter.split_documents(docs_pymupdf)
splits_4llm = text_splitter.split_documents(docs_4llm)
splits_vlm = text_splitter.split_documents(docs_vlm) if docs_vlm else []

print(f"PyMuPDF splits: {len(splits_pymupdf)}")
print(f"4LLM splits: {len(splits_4llm)}")
print(f"VLM splits: {len(splits_vlm)}")

PyMuPDF splits: 85
4LLM splits: 117
VLM splits: 65


In [39]:
# chunk_size를 바꿔가며 문서가 몇 개의 청크로 나뉘는지 확인
for size in [300, 700, 1500]:
    ts = RecursiveCharacterTextSplitter(
        chunk_size=size,  
        chunk_overlap=int(size * 0.2),  
        separators=["\n\n", "\n", ". ", ",", " "], 
        length_function=len,  
    )
    n_pymupdf = len(ts.split_documents(docs_pymupdf))  # PyMuPDF 문서를 청크로 나눈 개수
    n_vlm = len(ts.split_documents(docs_vlm)) if docs_vlm else 0  
    print(f"chunk_size={size:>5}, overlap={int(size*0.2):>4} → PyMuPDF: {n_pymupdf:>3}개, VLM: {n_vlm:>3}개")  

chunk_size=  300, overlap=  60 → PyMuPDF: 181개, VLM: 144개
chunk_size=  700, overlap= 140 → PyMuPDF:  80개, VLM:  63개
chunk_size= 1500, overlap= 300 → PyMuPDF:  41개, VLM:  27개


### Chunk_size가 작으면
- chunk_size가 작을수록 split(조각) 수가 많아져 검색 정밀도가 올라가지만,
- 너무 작으면 문맥이 단절되어 답변이 부정확해질 수 있습니다.


### Chunk_size가 크면
- chunk_size가 클수록 더 많은 문맥을 보존하고 의미를 잃지 않지만,
- 너무 크면 검색 결과가 부정확해지거나, 불필요한 정보까지 포함되어 Relevance가 떨어지고,
- LLM context 길이 초과(메모리/속도 저하) 위험도 있습니다.

### 적정 chunk_size
- 이번 실험에서는 적정 균형점을 찾아 chunk_size=700, overlap=180을 사용합니다.

## 2-3. Base RAG 세팅 (고정)

- chunk_size=700, chunk_overlap=180
- embeddings: BAAI/bge-m3
- top_k=20
- LLM: Qwen2.5-7B 계열 (로컬 Qwen fallback 설정)

In [40]:
# 임베딩 모델 설정
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",  # 사용할 임베딩 모델 이름
    model_kwargs={"device": "cpu"},  
    encode_kwargs={"normalize_embeddings": True},  
)
TOP_K = 5  # 검색 시 가져올 상위 문서 개수

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 32279.12it/s]


In [41]:
# ========== LLM과 RAG 프롬프트 정의 ==========
# 목적: Qwen2.5-VL 모델을 텍스트 기반 질의응답에 사용
# RAG_PROMPT: 검색된 맥락과 사용자 질문을 기반으로 답변 생성

# RAG 답변 생성용 프롬프트 템플릿
RAG_PROMPT = """다음 맥락을 바탕으로 질문에 답하세요. 맥락에 없는 내용은 추측하지 마세요.

맥락:
{context}

질문: {question}
답변:"""  # {context}와 {question}을 나중에 구체적인 값으로 대체


def generate_with_qwen(prompt: str, max_new_tokens: int = 256) -> str:
    """
    Qwen2.5-VL을 사용해 텍스트 프롬프트에 대한 답변 생성
    
    Args:
        prompt (str): 입력 프롬프트 (질문 포함)
        max_new_tokens (int): 생성할 최대 토큰 수
    
    Returns:
        str: 모델이 생성한 답변
    """
    # 메시지 포맷: 일반 텍스트 프롬프트만 사용 (이미지 없음)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},  # 텍스트 입력
            ],
        }
    ]
    
    # 프로세서를 이용해 메시지를 모델 입력 포맷으로 변환
    text = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(text=[text], return_tensors="pt").to(vlm_model.device)

    # 모델로 텍스트 생성
    with torch.inference_mode():  # 그래디언트 계산 불필요 (추론만 수행)
        generated_ids = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens)

    # 생성된 토큰 정리 (입력 부분 제거하고 출력만 추출)
    generated_ids_trimmed = generated_ids[:, inputs.input_ids.shape[1]:]
    
    # 토큰을 텍스트로 디코딩
    response = vlm_processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    return response.strip()  # 앞뒤 공백 제거 후 반환

In [42]:
# ========== RAG 평가 함수들 정의 ==========
# 목적: LLM 답변의 정확도, 근거성, 응답 시간을 평가하는 반복 가능한 함수들

import re  # 정규표현식 라이브러리 (숫자 패턴 추출용)

def check_accuracy(llm_answer: str, expected_answer: str) -> str:
    """
    LLM 답변이 정답과 얼마나 일치하는지 평가
    
    평가 등급:
    - 'exact': 정답이 답변에 정확히 포함됨
    - 'partial': 정답의 숫자 일부가 답변에 포함됨
    - 'fail': 전혀 일치하지 않음
    
    Args:
        llm_answer (str): LLM이 생성한 답변
        expected_answer (str): 기대하는 정답
    
    Returns:
        str: 정확도 등급 ('exact', 'partial', 'fail')
    """
    ans, exp = llm_answer.strip(), expected_answer.strip()  # 공백 제거 후 답변과 정답 저장
    
    if not ans: 
        return 'fail'  # 답변이 비어 있으면 실패
    
    if exp in ans: 
        return 'exact'  # 정답 문자열이 답변에 포함되면 정확 일치
    
    nums_exp = re.findall(r'[\d.,]+', exp)  # 정답에서 숫자 패턴 추출 (예: "16.4", "1,234")
    nums_ans = re.findall(r'[\d.,]+', ans)  # 답변에서 숫자 패턴 추출
    
    for n in nums_exp:  # 정답 숫자들을 하나씩 확인
        if any(n in a for a in nums_ans): 
            return 'partial'  # 숫자 일부라도 맞으면 부분 일치
    
    return 'fail'  # 그 외에는 실패


def check_grounded(retrieved_docs, expected_answer: str) -> tuple:
    """
    검색된 문서들에 정답이 실제로 포함되어 있는지 확인 (근거성 검증)
    
    Args:
        retrieved_docs (List[Document]): FAISS 검색 결과 문서 리스트
        expected_answer (str): 기대하는 정답
    
    Returns:
        tuple: (근거성 여부, 근거 스니펫)
            - (True, "...스니펫..."): 정답을 포함한 문서의 일부 반환
            - (False, ""): 어떤 문서에도 정답이 없음
    """
    exp = expected_answer.strip()  # 정답 문자열 공백 제거
    
    for doc in retrieved_docs:  # 검색된 문서들을 순회
        idx = doc.page_content.find(exp)  # 문서 안에 정답 문자열이 있는지 확인
        
        if idx != -1:  # 정답 문자열을 찾은 경우
            start = max(0, idx - 100)  # 앞쪽 문맥 100자 포함 (너무 앞으로 가지 않도록)
            end = min(len(doc.page_content), idx + len(exp) + 100)  # 뒤쪽 문맥 100자 포함
            snippet = doc.page_content[start:end]  # 근거 스니펫 추출
            return True, f"...{snippet}..."  # 근거가 있으므로 True와 함께 스니펫 반환
    
    return False, ""  # 어떤 문서에도 없으면 False 반환


def evaluate_rag(question, expected_answer, retrieved_docs, llm_answer, elapsed_sec):
    """
    RAG 시스템의 전체 성능 평가 (정확도 + 근거성 + 응답시간)
    
    Args:
        question (str): 사용자 질문
        expected_answer (str): 기대하는 정답
        retrieved_docs (List[Document]): FAISS 검색 결과 문서 리스트
        llm_answer (str): LLM이 생성한 답변
        elapsed_sec (float): 응답 소요 시간(초)
    
    Returns:
        dict: 평가 결과 딕셔너리
            - 'accuracy': 정확도 등급 ('exact'/'partial'/'fail')
            - 'grounded': 근거 존재 여부 (True/False)
            - 'grounded_evidence': 근거 스니펫
            - 'elapsed': 응답 소요 시간
    """
    grounded, evidence = check_grounded(retrieved_docs, expected_answer)  # 검색 문서에 근거가 있는지 확인
    
    return {
        "accuracy": check_accuracy(llm_answer, expected_answer),  # 답변 정확도 평가
        "grounded": grounded,  # 근거 존재 여부
        "grounded_evidence": evidence,  # 근거 스니펫
        "elapsed": elapsed_sec,  # 응답 소요 시간
    }

## 2-4. FAISS VectorStoreRetriever 기반 답변 비교

In [43]:
# ========== RAG 파이프라인 실행 함수 ==========
# 목적: 청크(splits) → FAISS 벡터스토어 → 검색 → LLM 답변 생성을 통합 함수화

def run_rag_on_splits(splits, name: str, num_questions: int = 3):
    """
    주어진 청크(splits)로 FAISS 벡터스토어를 구축하고, 
    테스트 질문들에 대해 RAG를 실행해 성능 평가
    
    Args:
        splits (List[Document]): 청크로 분할된 문서 리스트
        name (str): 실험 이름 (출력용, 예: "PyMuPDFLoader")
        num_questions (int): 테스트할 질문 개수 (기본값: 3)
    
    Returns:
        List[dict]: 각 질문별 평가 결과 리스트
    """
    # 청크가 없으면 실행하지 않음
    if not splits:  
        print(f"{name}: splits 없음. 건너뜀.")  # 스킵 메시지 출력
        return []  # 빈 결과 반환

    # Step 1: 청크들로 FAISS 벡터스토어 생성 (임베딩 변환 + 인덱싱)
    vs = FAISS.from_documents(splits, embeddings)  
    
    # Step 2: 검색 인터페이스 생성 (유사도 기반, top_k=5)
    retriever = vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})  
    
    results = []  # 평가 결과 저장용 리스트

    # Step 3: 각 질문에 대해 RAG 실행
    for t in test_questions[:num_questions]:  # 앞에서부터 지정한 개수만큼 질문 반복
        q, exp = t["question"], t["expected_answer"]  # 질문과 기대 정답 추출
        
        # Step 3-1: 질문과 유사한 문서 청크 검색 (상위 TOP_K개)
        docs = retriever.invoke(q)  
        
        # Step 3-2: 검색된 문서들을 하나의 맥락으로 결합
        context = "\n---\n".join(d.page_content for d in docs)  
        
        # Step 3-3: LLM으로 답변 생성
        t0 = time.time()  # 시작 시간 기록
        ans = generate_with_qwen(RAG_PROMPT.format(context=context, question=q))  # Qwen2.5-VL로 답변 생성
        elapsed = time.time() - t0  # 응답 소요 시간 계산

        # Step 3-4: 답변 평가 (정확도, 근거성, 시간)
        ev = evaluate_rag(q, exp, docs, ans, elapsed)
        
        # Step 3-5: 결과 저장
        results.append({
            "question": q, 
            "expected_answer": exp, 
            "answer": ans[:200],  # 답변 일부만 저장
            **ev  # 평가 결과 딕셔너리 펼치기
        })

    return results  # 전체 결과 반환

In [44]:
# ========== 실험 1: PyMuPDFLoader 추출본으로 RAG 실행 ==========
# 목적: 순수 텍스트 추출 방식이 RAG 성능에 미치는 영향 확인

res_pymupdf = run_rag_on_splits(splits_pymupdf, "PyMuPDFLoader", num_questions=12)

# 결과 상세 출력
for r in res_pymupdf:
    print(f"Q: {r['question']}")  # 질문 출력
    print(f"  정답: {r['expected_answer']}")  # 기대 정답 출력
    print(f"  LLM 답변: {r['answer']}")  # LLM이 생성한 답변 (처음 200자)
    # 평가 결과 출력 (정확도: exact/partial/fail, 근거성: True/False, 응답 시간)
    print(f"  정확도: {r['accuracy']}, 근거성: {r['grounded']}, 속도: {r['elapsed']:.2f}s")
    # 답변이 근거가 있는 경우에만 근거 스니펫 출력
    if r['grounded'] and r.get('grounded_evidence'):
        print(f"  근거: \"{r['grounded_evidence']}\"")
    print()

Q: 삼성전자 2025년 4분기 매출액은?
  정답: 93.8조원
  LLM 답변: 삼성전자의 2025년 4분기 매출액은 93.8조원입니다.
  정확도: exact, 근거성: True, 속도: 0.66s
  근거: "...김운호  6915-5656  
 
 
IBKS RESEARCH │7 
 2025년 4분기 매출액은 93.8조원 
삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. 
MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 가격..."

Q: 삼성전자 2025년 4분기 영업이익은?
  정답: 20.1조원
  LLM 답변: 삼성전자의 2025년 4분기 영업이익은 20.1조원입니다.
  정확도: exact, 근거성: True, 속도: 0.66s
  근거: "... 복합적 영향으로 부진했고, 디스
플레이는 중소형, 대형 모두 물량이 개선된 영향으로 증가했다. 삼성전자의 
2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원이다. 
DS, 디스플레이를 제외한 나머지 사업부 영업이익은 3분기 대비 감소하였
다. DS는 큰 폭의 가격 상승으로 2배 이상 증가했고, 디스플레이는 소형, 
대형 물량 증가로 ..."

Q: 삼성전자 DS 부문 2025년 4분기 영업이익은?
  정답: 16.4조원
  LLM 답변: 삼성전자의 DS 부문 2025년 4분기 영업이익은 16.4조원입니다.
  정확도: exact, 근거성: True, 속도: 0.70s
  근거: "... 증가, MX는 원가 부담 및 ASP 하락으로 47.6% 감소, VD/가전은 적자폭이 확
대되었다. 사업부별로는  
1) DS 사업부는 2025년 3분기 대비 134.4% 증가한 16.4조원이다. 메모리는 17.2
조원, 비메모리는 0.8조원 수준의 영업적자이다. DRAM, NAND 모두 ASP 상
승 영향을 크게 받을 것으로 분석된다. 비메모리는 물량 부진 및 충당..."

Q: 삼성전자 메모리 사업부 2025년

In [45]:
# ========== 실험 2: PyMuPDF4LLMLoader 추출본으로 RAG 실행 ==========
# 목적: Markdown 구조를 보존한 추출이 PyMuPDFLoader 대비 얼마나 개선되는지 확인

res_4llm = run_rag_on_splits(splits_4llm, "PyMuPDF4LLMLoader", num_questions=12)

# 결과 상세 출력
for r in res_4llm:
    print(f"Q: {r['question']}")  # 질문 출력
    print(f"  정답: {r['expected_answer']}")  # 기대 정답 출력
    print(f"  LLM 답변: {r['answer']}")  # LLM이 생성한 답변 (처음 200자)
    # 평가 결과 출력 (정확도, 근거성, 응답 시간)
    print(f"  정확도: {r['accuracy']}, 근거성: {r['grounded']}, 속도: {r['elapsed']:.2f}s")
    # 근거 스니펫 출력
    if r['grounded'] and r.get('grounded_evidence'):
        print(f"  근거: \"{r['grounded_evidence']}\"")
    print()

Q: 삼성전자 2025년 4분기 매출액은?
  정답: 93.8조원
  LLM 답변: 삼성전자의 2025년 4분기 매출액은 93.8조원입니다.
  정확도: exact, 근거성: True, 속도: 0.47s
  근거: "...김운호  6915-5656 

MX 제외 전 사업부 증가. DS가 주도 

## 2025년 4분기 매출액은 93.8조원 

삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 가격..."

Q: 삼성전자 2025년 4분기 영업이익은?
  정답: 20.1조원
  LLM 답변: 삼성전자의 2025년 4분기 영업이익은 20.1조원입니다.
  정확도: exact, 근거성: True, 속도: 0.49s
  근거: "...의 복합적 영향으로 부진했고, 디스 플레이는 중소형, 대형 모두 물량이 개선된 영향으로 증가했다. 삼성전자의 2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원이다. DS, 디스플레이를 제외한 나머지 사업부 영업이익은 3분기 대비 감소하였 다. DS는 큰 폭의 가격 상승으로 2배 이상 증가했고, 디스플레이는 소형, 대형 물량 증가로 66..."

Q: 삼성전자 DS 부문 2025년 4분기 영업이익은?
  정답: 16.4조원
  LLM 답변: 삼성전자의 DS 부문 2025년 4분기 영업이익은 16.4조원입니다.
  정확도: exact, 근거성: True, 속도: 0.54s
  근거: "...가, MX는 원가 부담 및 ASP 하락으로 47.6% 감소, VD/가전은 적자폭이 확 대되었다. 사업부별로는 

- 1) DS 사업부는 2025년 3분기 대비 134.4% 증가한 16.4조원이다. 메모리는 17.2 조원, 비메모리는 0.8조원 수준의 영업적자이다. DRAM, NAND 모두 ASP 상 승 영향을 크게 받을 것으로 분석된다. 비메모리는 물량 부진 및 충당..."

Q: 삼성전자 메모리 사업부 

In [46]:
# ========== 실험 3: VLM 추출본으로 RAG 실행 ==========
# 목적: VLM(시각언어모델)을 사용한 추출이 3가지 방식 중 최고 성능인지 확인

res_vlm = run_rag_on_splits(splits_vlm, "VLM", num_questions=12)

# 결과 상세 출력
for r in res_vlm:
    print(f"Q: {r['question']}")  # 질문 출력
    print(f"  정답: {r['expected_answer']}")  # 기대 정답 출력
    print(f"  LLM 답변: {r['answer']}")  # LLM이 생성한 답변 (처음 200자)
    # 평가 결과 출력 (정확도, 근거성, 응답 시간)
    print(f"  정확도: {r['accuracy']}, 근거성: {r['grounded']}, 속도: {r['elapsed']:.2f}s")
    # 근거 스니펫 출력
    if r['grounded'] and r.get('grounded_evidence'):
        print(f"  근거: \"{r['grounded_evidence']}\"")
    print()

Q: 삼성전자 2025년 4분기 매출액은?
  정답: 93.8조원
  LLM 답변: 삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원입니다.
  정확도: exact, 근거성: True, 속도: 0.73s
  근거: "...```markdown
## 2025년 4분기 매출액은 93.8조원

MX 제외 전 사업부 증가. DS가 주도

삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출..."

Q: 삼성전자 2025년 4분기 영업이익은?
  정답: 20.1조원
  LLM 답변: 삼성전자의 2025년 4분기 영업이익은 20.1조원입니다.
  정확도: exact, 근거성: True, 속도: 0.49s
  근거: "...의 복합적 영향으로 부진했고, 디스 플레이는 중소형, 대형 모두 물량이 개선된 영향으로 증가했다. 삼성전자의 2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원이다. DS, 디스플레이를 제외한 나머지 사업부 영업이익은 3분기 대비 감소하였 다. DS는 큰 폭의 가격 상승으로 2배 이상 증가했고, 디스플레이는 소형, 대형 물량 증가로 66..."

Q: 삼성전자 DS 부문 2025년 4분기 영업이익은?
  정답: 16.4조원
  LLM 답변: 삼성전자의 DS 부문 2025년 4분기 영업이익은 16.4조원입니다.
  정확도: exact, 근거성: True, 속도: 0.53s
  근거: "...7% 증가, MX는 원가 부담 및 ASP 하락으로 47.6% 감소, VD/가전은 적자폭이 확대되었다. 사업부별로는

1) DS 사업부는 2025년 3분기 대비 134.4% 증가한 16.4조원이다. 메모리는 17.2조원, 비메모리는 0.8조원 수준의 영업적자이다. DRAM, NAND 모두 ASP 상승 영향을 크게 받을 것으로 분석된다. 비메모리는 물량 부진 및 충당금으..."

Q: 삼성전자 메모리 사업부 2025년 4분

In [ ]:
# ========== 3종 추출본 splits 저장 ==========
# 목적: Part 3~5에서 다양한 검색 기법(하이브리드, Multi-Query, Reranking)을 
# 테스트하기 위해 3가지 추출 결과를 파일에 저장

split_save_filename = "extracted_chunk_splits.pkl"  # 더 직관적인 파일명 (splits 저장)

# 3가지 추출 방식의 청킹 결과를 딕셔너리로 저장
with open(split_save_filename, "wb") as f:
    pickle.dump({
        "pymupdf": splits_pymupdf,  # PyMuPDFLoader 방식
        "4llm": splits_4llm,  # PyMuPDF4LLMLoader 방식
        "vlm": splits_vlm  # VLM 방식
    }, f)

print(f"{split_save_filename} 저장 완료.")

extracted_chunk_splits.pkl 저장 완료.
